To create a CNN model which should be trained on 2 classes i.e cats and dogs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
 ## Importing all the libraries
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D , MaxPool2D , Flatten , Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import Adam

from sklearn.metrics import classification_report,confusion_matrix

import tensorflow as tf

import cv2
import os

import numpy as np

In [ ]:
# execute this only once... if you run again, skip this part
import zipfile
with zipfile.ZipFile("/content/drive/MyDrive/Arificial Intelegence Associate/CVE/5_train.zip", 'r') as zip_ref:        # get the path for train.zip using copypath. paste the link within the quotation
     # creates new folder 'cats_dogs_images' and saves all images present in train.zip
    zip_ref.extractall("/content/drive/MyDrive/Arificial Intelegence Associate/CVE/extracted_images")  # create a new folder, rename it and get the path of it using copypath.. paste the link here



In [ ]:
files_list = os.listdir("/content/drive/MyDrive/Arificial Intelegence Associate/CVE/extracted_images/train")

# Count the number of files
num_files = len(files_list)
print(num_files)

In [ ]:
print(files_list)

In [ ]:
cat = 0
dog = 0
for word in files_list:
  if "cat" in word.lower():
    cat = cat + 1
  elif "dog" in word.lower():
    dog = dog + 1

In [ ]:
print(cat)
print(dog)

In [ ]:
import os, shutil, pathlib

original_dir = pathlib.Path(r"/content/drive/MyDrive/Arificial Intelegence Associate/CVE/extracted_images/train") # get the path of train folder in cats_dogs_images using copypath and paste the link here
new_base_dir = pathlib.Path(r"/content/drive/MyDrive/Arificial Intelegence Associate/CVE/my_images") # create a new folder, rename it and get the path using copypath and paste the link here.


# define a function to create subsets for train, test and validation
def make_subset(subset_name, start_index, end_index):
    for category in ("cat", "dog"):
        dir = new_base_dir / subset_name / category
        os.makedirs(dir)
        fnames = [f"{category}.{i}.jpg" for i in range(start_index, end_index)]
        for fname in fnames:
            shutil.copyfile(src=original_dir / fname,
                            dst=dir / fname)

# calling the function thrice to create 3 subsets.
make_subset("train", start_index=0, end_index=1000)
make_subset("validation", start_index=1000, end_index=1500)
make_subset("test", start_index=1500, end_index=2500)

In [ ]:
pic = plt.imread("/content/my_images/test/dog/dog.1590.jpg")
plt.imshow(pic)

In [ ]:
a = plt.imread("/content/my_images/train/dog/dog.967.jpg")
plt.imshow(a)

In [ ]:
## This step helps in converting all the images present in folder to same shape.
import cv2
import numpy as np
labels = ['cat', 'dog']
img_size = 224
def get_data(data_dir):
    data = []
    for label in labels:
        path = os.path.join(data_dir, label) # path to cat folder inside train folder
        class_num = labels.index(label) # 0
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img)) #convert BGR to RGB format
                resized_arr = cv2.resize(img_arr, (224, 224)) # Reshaping images to preferred size
                data.append([resized_arr, class_num])
            except Exception as e:
                print(e)
    return np.array(data,dtype="object")

In [ ]:
train = get_data(r'/content/my_images/train')
val = get_data(r'/content/my_images/validation')



In [ ]:
type(train)
train.shape

In [ ]:
train[1500][0]

In [ ]:
train[1500][1]

In [ ]:
a = 1
b = 2
c = 3
# [[a,123],[b,213],3[c,1]]
myarr = np.array([[1,123],[2,213],[3,31]])
print(myarr)
print(myarr.shape)

## The numpy array is able to convert nested list into 2d np array because the element of nested list is of same data type

In [ ]:
image1 = np.array([[2,3],[4,5]])
image2 = np.array([[5,3],[7,4]])
image3 = np.array([[3,4],[8,8]])

np.array([[image1,0],[image2,0],[image3,0]])
## The numpy array is unable to convert nested list into 2d np array because the element of nested list is of different data type

In [ ]:
## Setting up every element inside the numpy array as Object to convert nested list into 2d np array
np.array([[image1,0],[image2,0],[image3,0]],dtype=object)

In [ ]:
np.array([[image1,0],[image2,0],[image3,0]],dtype=object).shape

In [ ]:
print(len(train))
print(len(val))
print(train.shape)
print(val.shape)

In [ ]:
type(0)

In [ ]:
print(type(train[1999][0]))
print(type(train[1999][1]))

In [ ]:
x_train = []
y_train = []
x_val = []
y_val = []

for feature, label in train:
  x_train.append(np.array(feature))
  y_train.append(int(label))

for feature, label in val:
  x_val.append(np.array(feature))
  y_val.append(int(label))

# Normalize the data
x_train = np.array(x_train).astype('float32') / 255
x_val = np.array(x_val).astype('float32') / 255

y_train = np.array(y_train)
y_val = np.array(y_val)


In [ ]:
## Generating data or images during runtime for training purpose.DO not do it for testing
datagen = ImageDataGenerator(
        featurewise_center=True,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range = 30,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.2, # Randomly zoom image
        width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
        horizontal_flip = True,  # randomly flip images
        vertical_flip=False)  # randomly flip images


datagen.fit(x_train)


In [ ]:
type(datagen)

In [ ]:
x_train.shape

In [ ]:
model = Sequential()
model.add(Conv2D(32,3,padding="same", activation="relu", input_shape=(224,224,3)))
model.add(MaxPool2D())

model.add(Conv2D(32, 3, padding="same", activation="relu"))
model.add(MaxPool2D())

model.add(Conv2D(64, 3, padding="same", activation="relu"))
model.add(MaxPool2D())
model.add(Dropout(0.4))

model.add(Flatten())
model.add(Dense(128,activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.summary()

In [ ]:
#tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = Adam(learning_rate=0.000001)
model.compile(optimizer = opt , loss = "binary_crossentropy" , metrics = ['accuracy'])

In [ ]:
type(x_train)

In [ ]:
type(y_train)

In [ ]:
#history = model.fit(x_train,y_train,epochs = 10 , validation_data = (x_val, y_val))


from keras.callbacks import ModelCheckpoint

# train the model
checkpointer = ModelCheckpoint(filepath='model.weights.best.keras', verbose=1, save_best_only=True)

hist = model.fit(datagen.flow(x_train, y_train, batch_size=32), epochs=100,
          validation_data=(x_val, y_val), callbacks=[checkpointer],
          verbose=2, shuffle=True)

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
loss = hist.history['loss']
val_loss = hist.history['val_loss']

epochs_range = range(100)

plt.figure(figsize=(15, 15))
plt.subplot(2, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
## Task
fine tune the configuration
train the model on more than 100
take a random picture of a dog then preprocess it accordingly and send it to machine for prediction and check your result